# Cloud Provider Analytics — Pipeline completo

```text
Landing → Bronze → Silver → Gold → Serving (AstraDB)
```

Orquestación end-to-end del MVP (parcial 2). La lógica vive en `src/jobs/`; este notebook ejecuta cada capa y muestra evidencias.

**Prerrequisitos:** dataset en `datalake/landing/`, keyspace `cloud_analytics` en consola Astra, token + bundle configurados.

In [ ]:
# Setup (Colab o local)
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q pyspark cassandra-driver
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/drive/MyDrive/cloud-provider-analytics")
    # os.environ["DATA_ROOT"] = "/content/datalake"
    # os.environ["ASTRA_DB_APPLICATION_TOKEN"] = "AstraCS:..."
    # os.environ["ASTRA_DB_SECURE_BUNDLE_PATH"] = "/content/secure-connect-cloud-analytics.zip"
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display
from pyspark.sql import SparkSession

from src.config import (
    BRONZE,
    CASSANDRA_KEYSPACE,
    CHECKPOINTS,
    CQL_DIR,
    DATA_ROOT,
    GOLD,
    LANDING,
    QUARANTINE,
    SILVER,
)
from src.cassandra.client import is_astra_configured
from src.jobs.bronze_streaming import (
    USAGE_EVENTS_BRONZE_PATH,
    USAGE_EVENTS_CHECKPOINT_PATH,
    USAGE_EVENTS_LANDING_GLOB,
)
from src.jobs.gold import ORG_DAILY_USAGE_BY_SERVICE
from src.jobs.silver import USAGE_EVENTS_QUARANTINE, USAGE_EVENTS_SILVER
from src.schemas.bronze_streaming import WATERMARK_DELAY

print(f"PROJECT_ROOT:      {PROJECT_ROOT}")
print(f"DATA_ROOT:         {DATA_ROOT}")
print(f"Astra configured:  {is_astra_configured()}")
print(f"KEYSPACE:          {CASSANDRA_KEYSPACE}")

In [ ]:
from src.spark.performance import configure_spark_performance

spark = (
    SparkSession.builder.appName("cloud-provider-analytics")
    .master("local[*]")
    .getOrCreate()
)
configure_spark_performance(spark)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

## 1. Batch Bronze — 3 maestros CSV

Ingesta de los 7 maestros CSV del landing → Parquet tipificado con `ingest_ts`, `source_file` y dedupe.

In [ ]:
from src.jobs.bronze_batch import run_batch_bronze, validate_bronze_uniqueness

batch_results = run_batch_bronze(spark)
display(pd.DataFrame(batch_results)[
    ["dataset_name", "raw_count", "deduped_count", "removed_duplicates", "written_count", "bronze_path"]
])

In [ ]:
for dataset in [
    "customers_orgs", "users", "billing_monthly",
    "resources", "support_tickets", "marketing_touches", "nps_surveys",
]:
    path = f"{BRONZE}/{dataset}"
    print(f"\n=== {dataset} ===")
    df = spark.read.parquet(path)
    df.printSchema()
    df.select(df.columns[:6]).show(5, truncate=False)

display(pd.DataFrame(validate_bronze_uniqueness(spark)))

## 2. Streaming Bronze — usage_events

Structured Streaming desde `usage_events_stream/*.jsonl` con watermark, dedupe `event_id`, late data y checkpoint.

**Late data (3 tiers):** watermark técnico → flag `is_late_arrival` (>10 min) → quarantine en Silver si >30 min.

Post-stream se ejecuta **reparquet** (`repartition` por `usage_date` + `service`) para optimizar lecturas por rango en Silver/Gold.

> Replay: `ingest_ts = event_ts` (watermark `60 days`). Producción: `ingest_ts = current_timestamp()` (watermark `10 minutes`).

In [ ]:
from src.jobs.bronze_streaming import run_streaming_bronze, validate_bronze_streaming

print(f"LANDING glob: {USAGE_EVENTS_LANDING_GLOB}")
print(f"CHECKPOINT:   {USAGE_EVENTS_CHECKPOINT_PATH}")
print(f"WATERMARK:    {WATERMARK_DELAY}")

streaming_result = run_streaming_bronze(spark, reset_state=True)
display(pd.DataFrame([{k: v for k, v in streaming_result.items() if k != "reparquet"}]))
print("Reparquet:", streaming_result.get("reparquet"))

In [ ]:
df = spark.read.parquet(USAGE_EVENTS_BRONZE_PATH)
df.printSchema()
df.select(
    "event_id", "event_ts", "service", "value", "schema_version",
    "carbon_kg", "genai_tokens", "source_file", "is_late_arrival"
).show(5, truncate=False)

display(pd.DataFrame([validate_bronze_streaming(spark)]))

## 3. Silver — 7 maestros + usage_events

Conformance de maestros, joins, features a **grano evento** (`usage_events`) y **grano org+servicio+fecha** (`org_service_daily` con `daily_cost_usd`), 3 métodos de anomalía y quarantine (incl. late extremo >30 min).

In [ ]:
from src.jobs.silver import run_silver, validate_silver

silver_results = run_silver(spark)
display(pd.DataFrame(silver_results))

events = next(r for r in silver_results if r["dataset_name"] == "usage_events")
display(pd.DataFrame([{
    "bronze": events["raw_count"],
    "silver_valid": events["valid_count"],
    "org_service_daily": events["org_service_daily_count"],
    "quarantine": events["quarantine_count"],
    "late_flagged": events["late_arrivals_flagged"],
    "late_quarantined": events["late_arrivals_quarantined"],
    "future_ts_quarantined": events["future_event_ts_quarantined"],
    "cost_anomalies_flagged": events["cost_anomalies_flagged"],
    "zscore": events["cost_anomalies_zscore"],
    "mad": events["cost_anomalies_mad"],
    "percentile": events["cost_anomalies_percentile"],
}]))
events["quarantine_sample"]

In [ ]:
from src.jobs.silver import ORG_SERVICE_DAILY_SILVER, USAGE_EVENTS_SILVER

df = spark.read.parquet(USAGE_EVENTS_SILVER)
df.select(
    "event_id", "org_name", "usage_date", "service",
    "cost_usd_increment", "requests", "cpu_hours", "storage_gb_hours",
    "genai_tokens", "carbon_kg", "org_user_count", "resource_region",
    "is_cost_anomaly", "is_cost_anomaly_zscore", "is_cost_anomaly_mad",
    "is_cost_anomaly_percentile",
).show(5, truncate=False)

daily = spark.read.parquet(ORG_SERVICE_DAILY_SILVER)
display(pd.DataFrame([{
    "org_service_daily_rows": daily.count(),
    "distinct_grain": daily.select("org_id", "usage_date", "service").distinct().count(),
}]))
daily.select(
    "org_id", "usage_date", "service", "daily_cost_usd",
    "requests", "event_count", "has_cost_anomaly",
).orderBy(daily.daily_cost_usd.desc()).show(5, truncate=False)

display(pd.DataFrame([validate_silver(spark)]))

## 4. Gold — marts de negocio

**Servidos (5):** FinOps, Soporte y GenAI → consultas CQL #1–#5.

**Gold-only (Parquet):** `cost_anomaly_mart`, `nps_by_org_date`, `marketing_touches_by_org_channel` (sin tabla Astra; ver `documentation/LOG_DECISIONES.md` §6).

In [ ]:
from src.jobs.gold import run_gold, validate_gold

gold_results = run_gold(spark)
display(pd.DataFrame(gold_results))

In [ ]:
df = spark.read.parquet(ORG_DAILY_USAGE_BY_SERVICE)
df.printSchema()
df.orderBy(df.total_daily_cost_usd.desc()).show(10, truncate=False)

display(pd.DataFrame([validate_gold(spark)]))

## 5. Serving — AstraDB (5 tablas, 5 consultas)

Carga de los 5 marts Gold vía **Structured Streaming `foreachBatch`** (consigna §5) y ejecución de consultas #1–#5. Requiere keyspace `cloud_analytics` en consola Astra + token + bundle.

In [ ]:
from pyspark.sql import functions as F
from src.jobs.serving_cassandra import run_serving

if not is_astra_configured():
    print("Skipping serving: set ASTRA_DB_APPLICATION_TOKEN and ASTRA_DB_SECURE_BUNDLE_PATH.")
else:
    top_org = (
        spark.read.parquet(ORG_DAILY_USAGE_BY_SERVICE)
        .groupBy("org_id")
        .agg(F.sum("total_daily_cost_usd").alias("cost"))
        .orderBy(F.desc("cost"))
        .first()
    )
    org_id = top_org["org_id"] if top_org else "org_rixa11dp"
    result = run_serving(spark, org_id=org_id)

    if result["load"]:
        display(pd.DataFrame(result["load"]))

    display(pd.DataFrame(result["query1_sample"]))
    display(pd.DataFrame(result["query2_top"]))
    display(pd.DataFrame(result["query3_sample"]))
    display(pd.DataFrame(result["query4_sample"]))
    display(pd.DataFrame(result["query5_sample"]))

## 6. Idempotencia — re-ejecución sin duplicados

Re-corremos **batch Bronze → streaming Bronze (sin `reset_state`) → Silver → Gold** y comparamos conteos antes/después.

| Capa | Estrategia |
|---|---|
| Bronze batch | `overwrite` + dedupe por clave natural |
| Bronze streaming | checkpoint + dedupe `event_id` (sin borrar checkpoint) |
| Silver / Gold | `overwrite` por dataset |
| Cassandra (opcional) | upsert por PK |

**Criterio OK:** mismos conteos y `event_id` únicos en Bronze/Silver; grano Gold estable.

In [ ]:
import os

from src.jobs.bronze_batch import run_batch_bronze
from src.jobs.bronze_streaming import run_streaming_bronze
from src.jobs.gold import (
    GENAI_TOKENS_BY_ORG_DATE,
    ORG_DAILY_USAGE_BY_SERVICE,
    REVENUE_BY_ORG_MONTH,
    TICKETS_BY_ORG_DATE,
    run_gold,
)
from src.jobs.silver import run_silver


def _lake_snapshot(spark) -> dict[str, int]:
    bronze_events = spark.read.parquet(USAGE_EVENTS_BRONZE_PATH)
    silver_valid = spark.read.parquet(USAGE_EVENTS_SILVER)
    gold_finops = spark.read.parquet(ORG_DAILY_USAGE_BY_SERVICE)

    quarantine_count = 0
    if os.path.isdir(USAGE_EVENTS_QUARANTINE):
        quarantine_count = spark.read.parquet(USAGE_EVENTS_QUARANTINE).count()

    master_total = sum(
        spark.read.parquet(f"{BRONZE}/{name}").count()
        for name in [
            "customers_orgs", "users", "billing_monthly", "resources",
            "support_tickets", "marketing_touches", "nps_surveys",
        ]
    )

    return {
        "bronze_masters": master_total,
        "bronze_events": bronze_events.count(),
        "bronze_events_distinct": bronze_events.select("event_id").distinct().count(),
        "silver_valid": silver_valid.count(),
        "silver_quarantine": quarantine_count,
        "gold_finops": gold_finops.count(),
        "gold_revenue": spark.read.parquet(REVENUE_BY_ORG_MONTH).count(),
        "gold_tickets": spark.read.parquet(TICKETS_BY_ORG_DATE).count(),
        "gold_genai": spark.read.parquet(GENAI_TOKENS_BY_ORG_DATE).count(),
    }


def _compare_idempotency(before: dict[str, int], after: dict[str, int]) -> pd.DataFrame:
    rows = []
    for key in before:
        b, a = before[key], after[key]
        rows.append({
            "metric": key,
            "before": b,
            "after": a,
            "delta": a - b,
            "ok": b == a,
        })
    return pd.DataFrame(rows)


before = _lake_snapshot(spark)
print("=== BEFORE re-run ===")
display(pd.DataFrame([before]))

# Re-run pipeline layers (streaming WITHOUT reset_state → checkpoint idempotency)
run_batch_bronze(spark)
streaming_rerun = run_streaming_bronze(spark, reset_state=False)
run_silver(spark)
run_gold(spark)

after = _lake_snapshot(spark)
print("=== AFTER re-run ===")
display(pd.DataFrame([after]))

comparison = _compare_idempotency(before, after)
display(comparison)

all_ok = comparison["ok"].all()
bronze_unique = after["bronze_events"] == after["bronze_events_distinct"]
silver_balance = (
    after["bronze_events"]
    == after["silver_valid"] + after["silver_quarantine"]
)

print(f"streaming_rerun written={streaming_rerun['written_count']} "
      f"distinct={streaming_rerun['distinct_event_ids']} "
      f"unique={streaming_rerun['is_unique']}")
print(f"idempotency [{'OK' if all_ok and bronze_unique and silver_balance else 'FAIL'}]: "
      f"counts_stable={all_ok} bronze_unique={bronze_unique} silver_balance={silver_balance}")

# Optional: Cassandra upsert idempotency (same row count per sample org)
if is_astra_configured():
    from src.cassandra.client import get_cassandra_session
    from src.cassandra.queries import TABLE_ORG_DAILY
    from src.jobs.serving_cassandra import load_org_daily_usage_by_service

    sample_org = (
        spark.read.parquet(ORG_DAILY_USAGE_BY_SERVICE)
        .select("org_id")
        .limit(1)
        .collect()[0]["org_id"]
    )
    session, cluster = get_cassandra_session()
    try:
        cassandra_before = session.execute(
            f"SELECT COUNT(*) FROM {TABLE_ORG_DAILY} WHERE org_id = %s",
            (sample_org,),
        ).one()[0]
        load_org_daily_usage_by_service(spark, session)
        cassandra_after = session.execute(
            f"SELECT COUNT(*) FROM {TABLE_ORG_DAILY} WHERE org_id = %s",
            (sample_org,),
        ).one()[0]
        display(pd.DataFrame([{
            "sample_org_id": sample_org,
            "cassandra_rows_before": cassandra_before,
            "cassandra_rows_after": cassandra_after,
            "cassandra_idempotent": cassandra_before == cassandra_after,
        }]))
    finally:
        cluster.shutdown()
else:
    print("Cassandra idempotency check skipped (Astra not configured).")